In [ ]:
# In[1]:

import pandas as pd
from pathlib import Path

# Setup
files_expected = {
    "metric.csv": "kpi_name",
    "trace.csv": "trace_name",
    "log.csv": "log_name",
    "error_logs.csv": None,  # no KPI column expected
}

loaded = {}
issues = []

# Load files if present, parse timestamp to UTC datetime
for fname, expected_kpi in files_expected.items():
    p = Path(fname)
    if not p.exists():
        loaded[fname] = None
        if expected_kpi is not None:  # only report missing for metric/trace/log per instructions
            issues.append({"file": fname, "issue": "file missing"})
        else:
            # still note missing error_logs if absent (for distinct cmdb list we need it)
            issues.append({"file": fname, "issue": "file missing"})
        continue
    try:
        df = pd.read_csv(fname)
    except Exception as e:
        loaded[fname] = None
        issues.append({"file": fname, "issue": f"read error: {e}"})
        continue
    # If timestamp column exists, parse as UTC datetime
    if "timestamp" in df.columns:
        try:
            df["timestamp"] = pd.to_datetime(df["timestamp"], unit="s", utc=True)
        except Exception as e:
            # If parsing fails, keep original but note
            issues.append({"file": fname, "issue": f"timestamp parse error: {e}"})
    loaded[fname] = df
    # Check emptiness
    if df.shape[0] == 0:
        if expected_kpi is not None:
            issues.append({"file": fname, "issue": "file is empty"})
        else:
            issues.append({"file": fname, "issue": "file is empty"})

# Helper to build summary for a file with expected KPI column name
def build_summary(df, fname, kpi_col):
    # return compact summary DataFrame as specified or None + record issues if problems
    if df is None:
        return None
    if kpi_col not in df.columns:
        issues.append({"file": fname, "issue": f"missing expected column '{kpi_col}'"})
        return None
    if "cmdb_id" not in df.columns:
        issues.append({"file": fname, "issue": "missing 'cmdb_id' column"})
        return None
    if "value" not in df.columns:
        issues.append({"file": fname, "issue": "missing 'value' column"})
        return None
    # Coerce value to numeric and drop NaNs for aggregation
    v = pd.to_numeric(df["value"], errors="coerce")
    df_valid = df.loc[v.notna()].copy()
    if df_valid.shape[0] == 0:
        issues.append({"file": fname, "issue": "no valid numeric 'value' points"})
        # return empty summary
        cols = ["source_file", "cmdb_id", "kpi_name", "count", "mean", "p50", "p90", "p95", "max"]
        return pd.DataFrame(columns=cols)
    df_valid["_v"] = pd.to_numeric(df_valid["value"], errors="coerce")
    grouped = (
        df_valid.groupby(["cmdb_id", kpi_col], observed=True)["_v"]
        .agg(
            count="count",
            mean="mean",
            p50=lambda x: x.quantile(0.5),
            p90=lambda x: x.quantile(0.9),
            p95=lambda x: x.quantile(0.95),
            max="max",
        )
        .reset_index()
    )
    # Normalize column names and add source_file
    grouped = grouped.rename(columns={kpi_col: "kpi_name"})
    grouped.insert(0, "source_file", fname)
    # Keep only requested columns and sort by count desc, limit top 20
    cols = ["source_file", "cmdb_id", "kpi_name", "count", "mean", "p50", "p90", "p95", "max"]
    grouped = grouped[cols].sort_values("count", ascending=False).head(20)
    # Round numeric columns to reasonable precision for compactness
    for c in ["mean", "p50", "p90", "p95", "max"]:
        if c in grouped.columns:
            grouped[c] = grouped[c].round(6)
    return grouped

# Build summaries for metric, trace, log
metric_summary = build_summary(loaded.get("metric.csv"), "metric.csv", "kpi_name")
trace_summary = build_summary(loaded.get("trace.csv"), "trace.csv", "trace_name")
log_summary = build_summary(loaded.get("log.csv"), "log.csv", "log_name")

# Ensure summaries are DataFrames (even empty) for consistent display
if metric_summary is None:
    metric_summary = pd.DataFrame(columns=["source_file", "cmdb_id", "kpi_name", "count", "mean", "p50", "p90", "p95", "max"])
if trace_summary is None:
    trace_summary = pd.DataFrame(columns=["source_file", "cmdb_id", "kpi_name", "count", "mean", "p50", "p90", "p95", "max"])
if log_summary is None:
    log_summary = pd.DataFrame(columns=["source_file", "cmdb_id", "kpi_name", "count", "mean", "p50", "p90", "p95", "max"])

# Distinct cmdb_id values across all four files (use available DataFrames)
cmdb_sets = set()
for fname, df in loaded.items():
    if df is None:
        continue
    if "cmdb_id" in df.columns:
        cmdb_sets.update(df["cmdb_id"].dropna().unique().tolist())
distinct_cmdbs = pd.DataFrame(sorted(cmdb_sets), columns=["cmdb_id"])

# Issues DataFrame (compact)
if len(issues) == 0:
    issues_df = pd.DataFrame([{"file": "none", "issue": "no issues detected"}])
else:
    issues_df = pd.DataFrame(issues)

# Final compact outputs (<=20 rows each)
metric_summary, trace_summary, log_summary, distinct_cmdbs.head(100), issues_df.head(100)

```
Out[1]:
```
```python
summary = (
    "Summary of telemetry load:\n"
    "- Files loaded: metric.csv, trace.csv, log.csv. error_logs.csv is present but empty.\n"
    "- No missing KPI columns were detected in metric/trace/log; each had valid 'value' and 'cmdb_id' columns.\n"
    "- Metric top groups (by point count = 25) include ts-voucher-service/socket (mean=4.0) and many ts-admin-* KPIs "
    "(cpu, mem, diskio, latency, socket, etc.).\n"
    "- Trace top groups (count = 25) include trace.to_ts-user-service.row_count (ts-user-service, mean≈10.72), several "
    "root->ts-admin-* traces, and many ts-travel2-service trace groups.\n"
    "- Log top groups (count = 25) show high row_count for services such as ts-basic-service (~1971.6), "
    "ts-travel2-service (~1879.0), ts-seat-service (~1481.2), ts-travel-service (~1349.6), with log.error_count ≈ 0 for most services.\n"
    "- Distinct cmdb_id values found across files: 69 unique IDs (includes 'root' and many ts-* services).\n"
    "- Conclusion: telemetry files are available and populated for metric/trace/log with consistent 25-point series; "
    "error_logs.csv contains no records."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(    source_file                      cmdb_id    kpi_name  count          mean           p50           p90           p95           max
372  metric.csv           ts-voucher-service      socket     25  4.000000e+00  4.000000e+00  4.000000e+00  4.000000e+00  4.000000e+00
0    metric.csv  ts-admin-basic-info-service         cpu     25  4.443280e-01  3.383960e-01  5.496290e-01  6.523230e-01  2.131947e+00
1    metric.csv  ts-admin-basic-info-service      diskio     25  8.239901e+05  4.714027e+05  1.769735e+06  1.772986e+06  1.793350e+06
2    metric.csv  ts-admin-basic-info-service       error     25  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
3    metric.csv  ts-admin-basic-info-service  latency-50     25  1.607500e-02  1.600900e-02  1.727900e-02  1.736100e-02  1.771100e-02
4    metric.csv  ts-admin-basic-info-service  latency-90     25  2.814600e-02  2.323600e-02  4.012600e-02  5.099600e-02  8.507900e-02
5    metric.csv  ts-admin-basic-info-service         mem     25  2.348429e+08  2.345800e+08  2.357119e+08  2.357133e+08  2.357426e+08
6    metric.csv  ts-admin-basic-info-service      socket     25  6.460570e+00  6.483333e+00  6.980000e+00  7.046667e+00  7.800000e+00
7    metric.csv  ts-admin-basic-info-service    workload     25  6.122400e-01  5.976500e-01  6.888800e-01  7.144000e-01  7.688500e-01
8    metric.csv       ts-admin-order-service         cpu     25  9.523100e-02  9.440300e-02  1.003640e-01  1.026690e-01  1.168490e-01
9    metric.csv       ts-admin-order-service      diskio     25  7.289266e+05  4.274452e+05  1.538898e+06  1.557636e+06  1.596524e+06
10   metric.csv       ts-admin-order-service         mem     25  1.904375e+08  1.904424e+08  1.904467e+08  1.904469e+08  1.904476e+08
11   metric.csv       ts-admin-order-service      socket     25  3.078667e+00  3.000000e+00  3.266667e+00  3.280000e+00  3.316667e+00
12   metric.csv       ts-admin-route-service         cpu     25  1.030210e-01  1.028160e-01  1.131550e-01  1.140480e-01  1.227220e-01
13   metric.csv       ts-admin-route-service      diskio     25  7.413512e+05  4.547170e+05  1.528484e+06  1.584093e+06  1.610144e+06
14   metric.csv       ts-admin-route-service         mem     25  1.890532e+08  1.890591e+08  1.890606e+08  1.890610e+08  1.890632e+08
15   metric.csv       ts-admin-route-service      socket     25  3.167733e+00  3.000000e+00  3.474000e+00  3.550000e+00  4.000000e+00
16   metric.csv      ts-admin-travel-service         cpu     25  4.309970e-01  3.405910e-01  6.292550e-01  8.479920e-01  1.204183e+00
17   metric.csv      ts-admin-travel-service      diskio     25  8.092423e+05  5.636635e+05  1.701109e+06  1.729945e+06  1.735313e+06
18   metric.csv      ts-admin-travel-service       error     25  1.333000e-03  0.000000e+00  0.000000e+00  0.000000e+00  3.331700e-02,     source_file             cmdb_id                                           kpi_name  count        mean         p50         p90         p95         max
775   trace.csv     ts-user-service                 trace.to_ts-user-service.row_count     25   10.720000   12.000000   15.200000   20.800000   26.000000
0     trace.csv                root  trace.to_ts-admin-basic-info-service.duration_...     25    0.011364    0.010027    0.012904    0.019419    0.026733
1     trace.csv                root  trace.to_ts-admin-basic-info-service.duration_p95     25    0.019223    0.011996    0.018916    0.062895    0.111055
2     trace.csv                root    trace.to_ts-admin-basic-info-service.error_rate     25    0.000000    0.000000    0.000000    0.000000    0.000000
3     trace.csv                root     trace.to_ts-admin-basic-info-service.row_count     25   17.640000   18.000000   20.000000   20.000000   23.000000
4     trace.csv                root     trace.to_ts-admin-travel-service.duration_mean     25    0.472734    0.633994    0.824246    0.854142    0.907341
5     trace.csv                root      trace.to_ts-admin-travel-service.duration_p95     25    0.726612    0.984217    1.369799    1.421858    1.566013
6     trace.csv                root        trace.to_ts-admin-travel-service.error_rate     25    0.000000    0.000000    0.000000    0.000000    0.000000
736   trace.csv  ts-travel2-service      trace.to_ts-order-other-service.duration_mean     25    0.006931    0.006638    0.008003    0.008199    0.010355
737   trace.csv  ts-travel2-service       trace.to_ts-order-other-service.duration_p95     25    0.012003    0.010757    0.014481    0.015921    0.023332
738   trace.csv  ts-travel2-service         trace.to_ts-order-other-service.error_rate     25    0.000000    0.000000    0.000000    0.000000    0.000000
739   trace.csv  ts-travel2-service          trace.to_ts-order-other-service.row_count     25   62.880000   65.000000   70.600000   72.600000   73.000000
740   trace.csv  ts-travel2-service            trace.to_ts-route-service.duration_mean     25    0.020103    0.023236    0.040174    0.042270    0.044036
741   trace.csv  ts-travel2-service             trace.to_ts-route-service.duration_p95     25    0.050558    0.065123    0.102293    0.102750    0.104516
742   trace.csv  ts-travel2-service               trace.to_ts-route-service.error_rate     25    0.000000    0.000000    0.000000    0.000000    0.000000
743   trace.csv  ts-travel2-service                trace.to_ts-route-service.row_count     25  477.680000  494.000000  536.200000  539.800000  558.000000
728   trace.csv  ts-travel2-service           trace.from_ts-seat-service.duration_mean     25    0.023003    0.017739    0.046215    0.048494    0.050163
729   trace.csv  ts-travel2-service            trace.from_ts-seat-service.duration_p95     25    0.046218    0.041344    0.105247    0.115224    0.163097
730   trace.csv  ts-travel2-service              trace.from_ts-seat-service.error_rate     25    0.000000    0.000000    0.000000    0.000000    0.000000
731   trace.csv  ts-travel2-service               trace.from_ts-seat-service.row_count     25  255.240000  265.000000  291.600000  295.600000  298.000000,    source_file                      cmdb_id         kpi_name  count     mean     p50     p90     p95   max
0      log.csv  ts-admin-basic-info-service  log.error_count     25     0.00     0.0     0.0     0.0     0
1      log.csv  ts-admin-basic-info-service    log.row_count     25    26.44    27.0    30.6    31.0    35
2      log.csv      ts-admin-travel-service  log.error_count     25     0.00     0.0     0.0     0.0     0
3      log.csv      ts-admin-travel-service    log.row_count     25    30.92    27.0    31.8    33.0   140
12     log.csv             ts-basic-service  log.error_count     25     0.00     0.0     0.0     0.0     0
13     log.csv             ts-basic-service    log.row_count     25  1971.56  2043.0  2238.2  2271.0  2361
11     log.csv              ts-auth-service    log.row_count     25   271.56   280.0   304.8   309.6   324
10     log.csv              ts-auth-service  log.error_count     25     0.00     0.0     0.0     0.0     0
21     log.csv           ts-consign-service    log.row_count     25    39.48    35.0    63.0    63.0    70
20     log.csv           ts-consign-service  log.error_count     25     0.00     0.0     0.0     0.0     0
28     log.csv          ts-food-map-service  log.error_count     25     0.00     0.0     0.0     0.0     0
29     log.csv          ts-food-map-service    log.row_count     25    17.24    18.0    21.6    24.4    25
85     log.csv            ts-travel-service    log.row_count     25  1349.56  1340.0  1685.4  1712.2  1716
84     log.csv            ts-travel-service  log.error_count     25     0.00     0.0     0.0     0.0     0
65     log.csv             ts-route-service    log.row_count     25   835.20   863.0   934.0   957.2  1156
64     log.csv             ts-route-service  log.error_count     25     0.00     0.0     0.0     0.0     0
67     log.csv              ts-seat-service    log.row_count     25  1481.24  1524.0  1679.2  1701.6  1778
66     log.csv              ts-seat-service  log.error_count     25     0.00     0.0     0.0     0.0     0
89     log.csv           ts-travel2-service    log.row_count     25  1879.04  1937.0  2118.6  2139.0  2304
88     log.csv           ts-travel2-service  log.error_count     25     0.00     0.0     0.0     0.0     0,                          cmdb_id
0                           root
1    ts-admin-basic-info-service
2         ts-admin-order-service
3         ts-admin-route-service
4        ts-admin-travel-service
..                           ...
64                 ts-user-mongo
65               ts-user-service
66  ts-verification-code-service
67              ts-voucher-mysql
68            ts-voucher-service

[69 rows x 1 columns],              file          issue
0  error_logs.csv  file is empty)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd

# Reuse loaded DataFrames from previous run if available
metric_df = loaded.get("metric.csv")
trace_df = loaded.get("trace.csv")
log_df = loaded.get("log.csv")

candidates = ["ts-auth-service", "ts-order-service", "ts-route-service", "ts-train-service", "ts-travel-service"]

# Helper to produce grouped summary for a dataframe given kpi column name and source label
def candidate_group_summary(df, fname, kpi_col, candidates, max_rows=100):
    cols_out = ["source_file", "cmdb_id", "kpi_name", "count", "mean", "p50", "p90", "p95", "p5", "min", "max"]
    if df is None:
        return pd.DataFrame(columns=cols_out)
    if "cmdb_id" not in df.columns or kpi_col not in df.columns or "value" not in df.columns:
        return pd.DataFrame(columns=cols_out)
    # Filter candidate cmdb_ids
    df_f = df[df["cmdb_id"].isin(candidates)].copy()
    if df_f.shape[0] == 0:
        return pd.DataFrame(columns=cols_out)
    # Ensure numeric
    df_f["_v"] = pd.to_numeric(df_f["value"], errors="coerce")
    df_f = df_f[df_f["_v"].notna()]
    if df_f.shape[0] == 0:
        return pd.DataFrame(columns=cols_out)
    grouped = (
        df_f.groupby(["cmdb_id", kpi_col], observed=True)["_v"]
        .agg(
            count="count",
            mean="mean",
            p50=lambda x: x.quantile(0.5),
            p90=lambda x: x.quantile(0.9),
            p95=lambda x: x.quantile(0.95),
            p5=lambda x: x.quantile(0.05),
            min="min",
            max="max",
        )
        .reset_index()
    )
    grouped = grouped.rename(columns={kpi_col: "kpi_name"})
    grouped.insert(0, "source_file", fname)
    # Sort by cmdb_id then kpi_name and limit rows
    grouped = grouped.sort_values(["cmdb_id", "kpi_name"]).head(max_rows)
    # Round numeric columns for compactness
    num_cols = ["mean", "p50", "p90", "p95", "p5", "min", "max"]
    for c in num_cols:
        if c in grouped.columns:
            grouped[c] = grouped[c].round(6)
    # Reorder to requested columns (some may be missing if df was empty)
    cols_present = [c for c in cols_out if c in grouped.columns]
    return grouped[cols_present]

# Build summaries
metric_res = candidate_group_summary(metric_df, "metric.csv", "kpi_name", candidates, max_rows=100)
trace_res  = candidate_group_summary(trace_df,  "trace.csv",  "trace_name", candidates, max_rows=100)
log_res    = candidate_group_summary(log_df,    "log.csv",    "log_name", candidates, max_rows=100)

# Presence listing for each candidate in each file
def presence_series(df, candidates):
    if df is None or "cmdb_id" not in df.columns:
        return {c: False for c in candidates}
    present = set(df["cmdb_id"].dropna().unique())
    return {c: (c in present) for c in candidates}

metric_presence = presence_series(metric_df, candidates)
trace_presence  = presence_series(trace_df,  candidates)
log_presence    = presence_series(log_df,    candidates)

presence_df = (
    pd.DataFrame({
        "cmdb_id": candidates,
        "metric_present": [metric_presence[c] for c in candidates],
        "trace_present":  [trace_presence[c]  for c in candidates],
        "log_present":    [log_presence[c]    for c in candidates],
    })
)

# Compact outputs (each <=100 rows). Return as variables per instructions.
metric_res, trace_res, log_res, presence_df

```
Out[2]:
```
```python
summary = (
    "Summary for the five candidate services (using full metric/trace/log series):\n\n"
    "1) Presence:\n"
    "- All five candidates (ts-auth-service, ts-order-service, ts-route-service, ts-train-service, ts-travel-service)\n"
    "  are present in metric.csv, trace.csv, and log.csv.\n\n"
    "2) Metric (filtered groups):\n"
    "- 39 grouped (cmdb_id, kpi_name) records for the five services were produced.\n"
    "- Examples / highlights:\n"
    "  * ts-auth-service: cpu mean ≈ 11.995, latency-90 mean ≈ 0.587s, mem ≈ 2.455e8.\n"
    "  * ts-travel-service: mem ≈ 2.711e8, socket mean ≈ 16.906, workload mean ≈ 7.471.\n"
    "  (All metric groups have count = 25 per series.)\n\n"
    "3) Trace (filtered groups):\n"
    "- 100 grouped rows returned (capped at 100). Many detailed trace features per service.\n"
    "- Examples:\n"
    "  * ts-auth-service: trace.from_root.duration_mean ≈ 0.280s, trace.from_root.row_count mean ≈ 67.84, trace.from_root.duration_p95 ≈ 0.633s.\n"
    "  * ts-travel-service: many 'from_' and 'to_' trace groups with varied durations and row_counts (multiple p95 values up to ~0.5–1.4s in some spans).\n\n"
    "4) Log (filtered groups):\n"
    "- Compact log summary rows for the five services (row_count and log.error_count).\n"
    "- Examples:\n"
    "  * ts-route-service: log.row_count mean ≈ 835.2 (min 143, max 1156), log.error_count = 0.\n"
    "  * ts-travel-service: log.row_count mean ≈ 1349.56 (min 380, max 1716), log.error_count = 0.\n"
    "  * ts-auth-service: log.row_count mean ≈ 271.56, log.error_count = 0.\n"
    "  * ts-order-service: log.row_count mean ≈ 295.96, log.error_count = 0.\n"
    "  * ts-train-service: only 3 log points (sparse) with log.row_count mean ≈ 58; log.error_count = 0.\n\n"
    "5) Key operational notes:\n"
    "- All candidate services have zero recorded log.error_count in these series.\n"
    "- ts-train-service log series is sparse (3 points) compared with others (25 points), which may affect comparisons.\n"
    "- Metric and trace series for these services are well populated (most groups show count = 25).\n\n"
    "If you want, I can: (a) show the filtered tables (head) for any one file, (b) highlight top anomalous KPIs, or (c) compute deltas across time windows."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(   source_file            cmdb_id    kpi_name  count          mean           p50           p90           p95            p5           min           max
0   metric.csv    ts-auth-service         cpu     25  1.199501e+01  1.184301e+01  1.317192e+01  1.342455e+01  1.083560e+01  9.739225e+00  1.398359e+01
1   metric.csv    ts-auth-service      diskio     25  5.118118e+04  4.994547e+04  5.804265e+04  5.996113e+04  4.365323e+04  3.862706e+04  6.347966e+04
2   metric.csv    ts-auth-service  latency-50     25  2.144900e-01  2.148920e-01  2.273240e-01  2.289940e-01  2.023420e-01  2.016620e-01  2.294470e-01
3   metric.csv    ts-auth-service  latency-90     25  5.874080e-01  5.145900e-01  7.408800e-01  7.935290e-01  4.386270e-01  4.330610e-01  1.203616e+00
4   metric.csv    ts-auth-service         mem     25  2.455126e+08  2.451347e+08  2.464232e+08  2.464970e+08  2.445685e+08  2.445020e+08  2.469550e+08
..         ...                ...         ...    ...           ...           ...           ...           ...           ...           ...           ...
34  metric.csv  ts-travel-service  latency-50     25  2.142700e-02  1.946800e-02  2.653400e-02  2.785000e-02  1.668200e-02  1.643400e-02  3.039000e-02
35  metric.csv  ts-travel-service  latency-90     25  3.752100e-01  2.895500e-01  6.905000e-01  7.071860e-01  9.142900e-02  9.135700e-02  7.214580e-01
36  metric.csv  ts-travel-service         mem     25  2.711318e+08  2.710697e+08  2.714445e+08  2.714688e+08  2.707765e+08  2.704248e+08  2.714786e+08
37  metric.csv  ts-travel-service      socket     25  1.690587e+01  1.665000e+01  1.872000e+01  1.910667e+01  1.443000e+01  1.325000e+01  2.036667e+01
38  metric.csv  ts-travel-service    workload     25  7.470930e+00  7.486733e+00  8.609277e+00  8.820523e+00  5.513647e+00  5.305433e+00  1.191491e+01

[39 rows x 11 columns],    source_file            cmdb_id                                      kpi_name  count       mean        p50        p90        p95         p5        min        max
0    trace.csv    ts-auth-service                 trace.from_root.duration_mean     25   0.280476   0.261763   0.334643   0.371689   0.234933   0.233188   0.418416
1    trace.csv    ts-auth-service                  trace.from_root.duration_p95     25   0.633396   0.502643   0.988385   1.094932   0.408591   0.405466   1.390859
2    trace.csv    ts-auth-service                    trace.from_root.error_rate     25   0.000000   0.000000   0.000000   0.000000   0.000000   0.000000   0.000000
3    trace.csv    ts-auth-service                     trace.from_root.row_count     25  67.840000  70.000000  76.200000  77.800000  58.400000  10.000000  81.000000
4    trace.csv    ts-auth-service      trace.from_ts-auth-service.duration_mean     25   0.002808   0.002500   0.004200   0.004917   0.001559   0.001388   0.005060
..         ...                ...                                           ...    ...        ...        ...        ...        ...        ...        ...        ...
95   trace.csv  ts-travel-service  trace.from_ts-admin-travel-service.row_count     25   8.840000   9.000000  10.000000  10.800000   7.000000   2.000000  11.000000
96   trace.csv  ts-travel-service      trace.from_ts-food-service.duration_mean     25   0.072260   0.022528   0.170374   0.180632   0.009577   0.009522   0.191589
97   trace.csv  ts-travel-service       trace.from_ts-food-service.duration_p95     25   0.214451   0.034642   0.509843   0.539410   0.010916   0.010686   0.673669
98   trace.csv  ts-travel-service         trace.from_ts-food-service.error_rate     25   0.000000   0.000000   0.000000   0.000000   0.000000   0.000000   0.000000
99   trace.csv  ts-travel-service          trace.from_ts-food-service.row_count     25  17.240000  18.000000  21.600000  24.400000  10.600000   2.000000  25.000000

[100 rows x 11 columns],   source_file            cmdb_id         kpi_name  count     mean     p50     p90     p95      p5  min   max
0     log.csv    ts-auth-service  log.error_count     25     0.00     0.0     0.0     0.0     0.0    0     0
1     log.csv    ts-auth-service    log.row_count     25   271.56   280.0   304.8   309.6   233.6   40   324
2     log.csv   ts-order-service  log.error_count     25     0.00     0.0     0.0     0.0     0.0    0     0
3     log.csv   ts-order-service    log.row_count     25   295.96   314.0   366.2   371.4   222.8   83   374
4     log.csv   ts-route-service  log.error_count     25     0.00     0.0     0.0     0.0     0.0    0     0
5     log.csv   ts-route-service    log.row_count     25   835.20   863.0   934.0   957.2   748.0  143  1156
6     log.csv   ts-train-service  log.error_count      3     0.00     0.0     0.0     0.0     0.0    0     0
7     log.csv   ts-train-service    log.row_count      3    58.00     1.0   137.8   154.9     1.0    1   172
8     log.csv  ts-travel-service  log.error_count     25     0.00     0.0     0.0     0.0     0.0    0     0
9     log.csv  ts-travel-service    log.row_count     25  1349.56  1340.0  1685.4  1712.2  1054.0  380  1716,              cmdb_id  metric_present  trace_present  log_present
0    ts-auth-service            True           True         True
1   ts-order-service            True           True         True
2   ts-route-service            True           True         True
3   ts-train-service            True           True         True
4  ts-travel-service            True           True         True)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

import numpy as np
import pandas as pd
from pandas.api import types as ptypes

# Reuse previously loaded DataFrames
metric_df = loaded.get("metric.csv")
trace_df = loaded.get("trace.csv")
log_df = loaded.get("log.csv")

candidates = ["ts-auth-service", "ts-order-service", "ts-route-service", "ts-train-service", "ts-travel-service"]

# Incident window (UTC)
window_start = pd.to_datetime("2024-01-25 10:31:10", utc=True)
window_end   = pd.to_datetime("2024-01-25 11:01:10", utc=True)

def detect_anomalies_for_file(df, source_file, kpi_col):
    cols_out = [
        "source_file", "cmdb_id", "kpi_name", "p95", "p5",
        "anomaly_count_in_window", "earliest_anomaly_timestamp",
        "max_value_in_window", "min_value_in_window", "max_severity_ratio"
    ]
    if df is None:
        return pd.DataFrame(columns=cols_out)
    # Check required columns
    for c in ("cmdb_id", "value", "timestamp", kpi_col):
        if c not in df.columns:
            return pd.DataFrame(columns=cols_out)
    # Ensure timestamp is datetime UTC
    df = df.copy()
    if not (ptypes.is_datetime64_any_dtype(df["timestamp"]) or ptypes.is_datetime64tz_dtype(df["timestamp"])):
        df["timestamp"] = pd.to_datetime(df["timestamp"], unit="s", utc=True)
    else:
        # If tz-naive, localize to UTC; if tz-aware, convert to UTC
        if ptypes.is_datetime64_any_dtype(df["timestamp"]) and not ptypes.is_datetime64tz_dtype(df["timestamp"]):
            df["timestamp"] = df["timestamp"].dt.tz_localize("UTC")
        else:
            df["timestamp"] = df["timestamp"].dt.tz_convert("UTC")
    # Filter candidate components
    df_cand = df[df["cmdb_id"].isin(candidates)].copy()
    if df_cand.shape[0] == 0:
        return pd.DataFrame(columns=cols_out)
    # Coerce numeric values
    df_cand["_v"] = pd.to_numeric(df_cand["value"], errors="coerce")
    df_cand = df_cand[df_cand["_v"].notna()].copy()
    if df_cand.shape[0] == 0:
        return pd.DataFrame(columns=cols_out)
    # Compute global thresholds (p95, p5) per (cmdb_id, kpi_col) using full series BEFORE filtering by time
    thresholds = (
        df_cand.groupby(["cmdb_id", kpi_col], observed=True)["_v"]
        .agg(p95=lambda x: x.quantile(0.95), p5=lambda x: x.quantile(0.05))
        .reset_index()
        .rename(columns={kpi_col: "kpi_name"})
    )
    # Filter original df_cand to incident window
    window_df = df_cand[(df_cand["timestamp"] >= window_start) & (df_cand["timestamp"] <= window_end)].copy()
    if window_df.shape[0] == 0:
        return pd.DataFrame(columns=cols_out)
    window_df = window_df.rename(columns={kpi_col: "kpi_name"})
    # Merge thresholds into window rows
    merged = window_df.merge(thresholds, on=["cmdb_id", "kpi_name"], how="left")
    # Drop rows without thresholds or with NaN thresholds
    merged = merged[merged["p95"].notna() & merged["p5"].notna()].copy()
    if merged.shape[0] == 0:
        return pd.DataFrame(columns=cols_out)
    # Identify anomalies per row
    merged["is_high"] = merged["_v"] >= merged["p95"]
    merged["is_low"]  = merged["_v"] <= merged["p5"]
    merged["is_anom"] = merged["is_high"] | merged["is_low"]
    anomalies = merged[merged["is_anom"]].copy()
    if anomalies.shape[0] == 0:
        return pd.DataFrame(columns=cols_out)
    # For severity ratios, compute safe ratios avoiding division by zero
    anomalies["high_ratio"] = np.where(
        anomalies["p95"] == 0,
        np.where(anomalies["_v"] > 0, np.inf, 0.0),
        anomalies["_v"] / anomalies["p95"]
    )
    anomalies["low_ratio"] = np.where(
        anomalies["_v"] == 0,
        np.where(anomalies["p5"] > 0, np.inf, 0.0),
        anomalies["p5"] / anomalies["_v"]
    )
    # Group anomalies and compute required summary
    grp = anomalies.groupby(["cmdb_id", "kpi_name"], observed=True).agg(
        p95=pd.NamedAgg(column="p95", aggfunc="first"),
        p5=pd.NamedAgg(column="p5", aggfunc="first"),
        anomaly_count_in_window=pd.NamedAgg(column="_v", aggfunc="count"),
        earliest_anomaly_timestamp=pd.NamedAgg(column="timestamp", aggfunc="min"),
        max_value_in_window=pd.NamedAgg(column="_v", aggfunc="max"),
        min_value_in_window=pd.NamedAgg(column="_v", aggfunc="min"),
        max_high_ratio=pd.NamedAgg(column="high_ratio", aggfunc="max"),
        max_low_ratio=pd.NamedAgg(column="low_ratio", aggfunc="max"),
    ).reset_index()
    grp["source_file"] = source_file
    # Compute max_severity_ratio per rules
    grp["max_severity_ratio"] = grp[["max_high_ratio", "max_low_ratio"]].max(axis=1)
    # Format earliest timestamp as ISO UTC string
    grp["earliest_anomaly_timestamp"] = grp["earliest_anomaly_timestamp"].dt.strftime("%Y-%m-%dT%H:%M:%SZ")
    # Select and order columns
    result = grp[[
        "source_file", "cmdb_id", "kpi_name", "p95", "p5",
        "anomaly_count_in_window", "earliest_anomaly_timestamp",
        "max_value_in_window", "min_value_in_window", "max_severity_ratio"
    ]]
    # Round numeric columns for compactness, keep inf as-is
    def safe_round(v):
        if pd.isna(v):
            return np.nan
        if np.isinf(v):
            return np.inf
        return np.round(v, 6)
    for c in ["p95", "p5", "max_value_in_window", "min_value_in_window", "max_severity_ratio"]:
        result[c] = result[c].apply(safe_round)
    return result

# Run detection
metric_anoms = detect_anomalies_for_file(metric_df, "metric.csv", "kpi_name")
trace_anoms  = detect_anomalies_for_file(trace_df,  "trace.csv",  "trace_name")
log_anoms    = detect_anomalies_for_file(log_df,    "log.csv",    "log_name")

# Combine results and select top 20 by anomaly_count_in_window descending
all_anoms = pd.concat([metric_anoms, trace_anoms, log_anoms], ignore_index=True)
if all_anoms.shape[0] == 0:
    final_result = pd.DataFrame([{"message": "No anomalies found in the incident window for the candidate services."}])
else:
    final_result = all_anoms.sort_values("anomaly_count_in_window", ascending=False).head(20)
    final_result = final_result[[
        "source_file", "cmdb_id", "kpi_name", "p95", "p5",
        "anomaly_count_in_window", "earliest_anomaly_timestamp",
        "max_value_in_window", "min_value_in_window", "max_severity_ratio"
    ]]

# Return compact outputs
metric_anoms.shape, trace_anoms.shape, log_anoms.shape, final_result.head(20)

```
Out[3]:
```
Summary of anomaly detection for the five candidate services in the incident window (2024-01-25 10:31:10 → 2024-01-25 11:01:10 UTC):

- Overall counts found:
  - Metric groups with anomalies: 39
  - Trace groups with anomalies: 136
  - Log groups with anomalies: 10
  - (185 anomalous groups in total; top 20 shown in the result.)

- Key pattern observed:
  - Most detected "anomalies" are for error / error_rate / log.error_count time series where the global p95 and p5 thresholds are both 0.0. Because the rule flagged points with value >= p95 OR value <= p5, series with p95==p5==0 cause all zero-valued points to be reported as anomalies (anomaly_count_in_window = 25 in many groups). Earliest anomaly timestamps in the output are mostly 2024-01-25T10:34:00Z.
  - Example groups flagged this way: many trace.*.error_rate series (auth/order/route/train/travel), metric "error" for ts-order-service and ts-travel-service, and log.error_count for route/auth/order/travel services. Those groups mostly have max = 0 and min = 0 in the window.
  - A few groups show small non-zero values above zero threshold (e.g., metric ts-travel-service "error" has max ≈ 0.04258). For groups where p5==0 and some anomaly values >0, the severity ratio calculation produced infinite (inf) ratios because dividing by zero threshold is undefined.

- Interpretation / caveats:
  - The majority of flagged anomalies are artifacts of zero-valued thresholds (p95==p5==0), not meaningful spikes in resource KPIs (CPU/memory/latency). They do not indicate clear resource degradation.
  - No clear anomalous increases in CPU, memory, disk I/O, or latency were indicated among these services in the incident window by this thresholding method.
  - The only potentially noteworthy signals are small non-zero error values in a few metric/error series (e.g., ts-travel-service metric "error"); these merit manual inspection.

- Recommended next steps:
  1. Inspect the raw time series for these error/error_rate/log.error_count groups to confirm whether any non-zero error events occurred and whether they are actionable.
  2. Change anomaly rules to avoid treating zero thresholds as anomalies (e.g., require thresholds > 0, use strict > / < instead of >=/<=, or require value to exceed a minimum absolute delta above baseline).
  3. If you want, I can show the raw windowed values for the small non-zero groups (e.g., ts-travel-service metric "error") or recompute anomalies using a non-zero minimum threshold to reduce false positives.

The original code execution output of IPython Kernel is also provided below for reference:

((39, 10), (136, 10), (10, 10),     source_file            cmdb_id                                   kpi_name  p95   p5  anomaly_count_in_window earliest_anomaly_timestamp  max_value_in_window  min_value_in_window  max_severity_ratio
9    metric.csv   ts-order-service                                      error  0.0  0.0                       25       2024-01-25T10:34:00Z              0.00000                  0.0                 0.0
41    trace.csv    ts-auth-service                 trace.from_root.error_rate  0.0  0.0                       25       2024-01-25T10:34:00Z              0.00000                  0.0                 0.0
45    trace.csv    ts-auth-service      trace.from_ts-auth-service.error_rate  0.0  0.0                       25       2024-01-25T10:34:00Z              0.00000                  0.0                 0.0
53    trace.csv   ts-order-service                 trace.from_root.error_rate  0.0  0.0                       25       2024-01-25T10:34:00Z              0.00000                  0.0                 0.0
49    trace.csv    ts-auth-service        trace.to_ts-auth-service.error_rate  0.0  0.0                       25       2024-01-25T10:34:00Z              0.00000                  0.0                 0.0
33   metric.csv  ts-travel-service                                      error  0.0  0.0                       25       2024-01-25T10:34:00Z              0.04258                  0.0                 inf
165   trace.csv  ts-travel-service  trace.to_ts-ticketinfo-service.error_rate  0.0  0.0                       25       2024-01-25T10:34:00Z              0.00000                  0.0                 0.0
157   trace.csv  ts-travel-service       trace.to_ts-route-service.error_rate  0.0  0.0                       25       2024-01-25T10:34:00Z              0.00000                  0.0                 0.0
153   trace.csv  ts-travel-service       trace.to_ts-order-service.error_rate  0.0  0.0                       25       2024-01-25T10:34:00Z              0.00000                  0.0                 0.0
179     log.csv   ts-route-service                            log.error_count  0.0  0.0                       25       2024-01-25T10:34:00Z              0.00000                  0.0                 0.0
125   trace.csv   ts-train-service       trace.to_ts-train-service.error_rate  0.0  0.0                       25       2024-01-25T10:34:00Z              0.00000                  0.0                 0.0
101   trace.csv   ts-route-service   trace.from_ts-travel2-service.error_rate  0.0  0.0                       25       2024-01-25T10:34:00Z              0.00000                  0.0                 0.0
97    trace.csv   ts-route-service    trace.from_ts-travel-service.error_rate  0.0  0.0                       25       2024-01-25T10:34:00Z              0.00000                  0.0                 0.0
113   trace.csv   ts-train-service     trace.from_ts-train-service.error_rate  0.0  0.0                       25       2024-01-25T10:34:00Z              0.00000                  0.0                 0.0
117   trace.csv   ts-train-service    trace.from_ts-travel-service.error_rate  0.0  0.0                       25       2024-01-25T10:34:00Z              0.00000                  0.0                 0.0
109   trace.csv   ts-train-service     trace.from_ts-basic-service.error_rate  0.0  0.0                       25       2024-01-25T10:34:00Z              0.00000                  0.0                 0.0
105   trace.csv   ts-route-service       trace.to_ts-route-service.error_rate  0.0  0.0                       25       2024-01-25T10:34:00Z              0.00000                  0.0                 0.0
89    trace.csv   ts-route-service     trace.from_ts-basic-service.error_rate  0.0  0.0                       25       2024-01-25T10:34:00Z              0.00000                  0.0                 0.0
93    trace.csv   ts-route-service     trace.from_ts-route-service.error_rate  0.0  0.0                       25       2024-01-25T10:34:00Z              0.00000                  0.0                 0.0
85    trace.csv   ts-order-service     trace.to_ts-station-service.error_rate  0.0  0.0                       25       2024-01-25T10:34:00Z              0.00000                  0.0                 0.0)```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import itertools

# Reuse metric_df from kernel state
# Ensure metric_df exists
assert 'metric_df' in globals(), "metric_df not found in kernel state"
candidates = ["ts-auth-service", "ts-order-service", "ts-route-service", "ts-train-service", "ts-travel-service"]

# Incident window (UTC)
window_start = pd.to_datetime("2024-01-25 10:31:10", utc=True)
window_end   = pd.to_datetime("2024-01-25 11:01:10", utc=True)

# Ensure timestamp is datetime UTC and value numeric
if not pd.api.types.is_datetime64_any_dtype(metric_df["timestamp"]):
    metric_df["timestamp"] = pd.to_datetime(metric_df["timestamp"], unit="s", utc=True)
else:
    # if tz-naive, localize; if tz-aware, convert to UTC
    if metric_df["timestamp"].dt.tz is None:
        metric_df["timestamp"] = metric_df["timestamp"].dt.tz_localize("UTC")
    else:
        metric_df["timestamp"] = metric_df["timestamp"].dt.tz_convert("UTC")

metric_df["_v"] = pd.to_numeric(metric_df["value"], errors="coerce")

# Compute global thresholds (p95, p90) for each (cmdb_id, kpi_name) using FULL series (no time filtering)
thresh_df = (
    metric_df[metric_df["cmdb_id"].isin(candidates) & metric_df["_v"].notna()]
    .groupby(["cmdb_id", "kpi_name"], observed=True)["_v"]
    .agg(p95=lambda x: x.quantile(0.95), p90=lambda x: x.quantile(0.90))
    .reset_index()
)

# Exclude groups where global p95 == 0 and exclude kpi_name containing 'error' or 'error_count' (case-insensitive)
mask_exclude = (thresh_df["p95"] == 0) | (thresh_df["kpi_name"].str.contains("error", case=False, na=False)) | (thresh_df["kpi_name"].str.contains("error_count", case=False, na=False))
thresh_filtered = thresh_df[~mask_exclude].copy()

# Prepare windowed points for candidate services
window_df = metric_df[
    (metric_df["cmdb_id"].isin(candidates)) &
    (metric_df["timestamp"] >= window_start) &
    (metric_df["timestamp"] <= window_end)
].copy()
# normalize kpi name column to match grouping
window_df = window_df.rename(columns={"kpi_name": "kpi_name"})

# For each remaining group compute required metrics using global thresholds (precomputed)
results = []
for _, row in thresh_filtered.iterrows():
    cmdb = row["cmdb_id"]
    kpi = row["kpi_name"]
    p95 = row["p95"]
    p90 = row["p90"]
    # select window points for this group
    grp = window_df[(window_df["cmdb_id"] == cmdb) & (window_df["kpi_name"] == kpi)].copy()
    # sort by timestamp
    grp = grp.sort_values("timestamp")
    vals = grp["_v"].to_numpy()
    times = grp["timestamp"].to_numpy()
    # counts
    anomaly_count_p95 = int(np.sum(vals >= p95)) if len(vals) > 0 else 0
    anomaly_count_p90 = int(np.sum(vals >= p90)) if len(vals) > 0 else 0
    # earliest anomaly ts for p90
    if anomaly_count_p90 > 0:
        idxs = np.where(vals >= p90)[0]
        earliest_anom_ts_p90 = pd.to_datetime(times[idxs[0]]).tz_convert("UTC").strftime("%Y-%m-%dT%H:%M:%SZ")
    else:
        earliest_anom_ts_p90 = None
    # max/min in window
    max_value_in_window = float(np.nan) if len(vals) == 0 else float(np.nanmax(vals))
    min_value_in_window = float(np.nan) if len(vals) == 0 else float(np.nanmin(vals))
    # longest consecutive run >= p90
    longest_run = 0
    if len(vals) > 0:
        mask = (vals >= p90).astype(int)
        # use itertools.groupby to find consecutive ones
        runs = [sum(1 for _ in group) for val, group in itertools.groupby(mask) if val == 1]
        longest_run = int(max(runs)) if len(runs) > 0 else 0
    # collect row
    results.append({
        "cmdb_id": cmdb,
        "kpi_name": kpi,
        "p95": p95,
        "p90": p90,
        "anomaly_count_p95": anomaly_count_p95,
        "anomaly_count_p90": anomaly_count_p90,
        "earliest_anomaly_ts_p90": earliest_anom_ts_p90,
        "max_value_in_window": max_value_in_window,
        "min_value_in_window": min_value_in_window,
        "longest_consecutive_run_p90": longest_run,
    })

res_df = pd.DataFrame(results)

if res_df.shape[0] == 0:
    # explicit empty list message as DataFrame
    final_result = pd.DataFrame([{"message": "No KPI groups passed the filters (p95>0 and excluding 'error'/'error_count')."}])
else:
    # sort by longest_consecutive_run_p90 desc, then anomaly_count_p90 desc, limit 20
    final_result = (
        res_df.sort_values(["longest_consecutive_run_p90", "anomaly_count_p90"], ascending=[False, False])
        .head(20)
    )
    # round numeric columns for compactness
    for c in ["p95", "p90", "max_value_in_window", "min_value_in_window"]:
        if c in final_result.columns:
            final_result[c] = final_result[c].round(6)
    # ensure column order
    final_result = final_result[[
        "cmdb_id", "kpi_name", "p95", "p90",
        "anomaly_count_p95", "anomaly_count_p90", "earliest_anomaly_ts_p90",
        "max_value_in_window", "min_value_in_window", "longest_consecutive_run_p90"
    ]]

final_result

```
Out[4]:
```
Summary (metric.csv, candidate services, incident window 2024-01-25 10:31:10 → 2024-01-25 11:01:10 UTC; thresholds computed from full series):

Overall
- 18 (cmdb_id, kpi_name) metric groups passed the filters (global p95 > 0 and excluding names with "error"/"error_count").
- Many groups show short spikes that exceed the global p90/p95 thresholds in the incident window. Most anomalies are transient (3 or fewer consecutive points above p90).

Top groups by longest consecutive run >= p90 (highest priority)
(only groups with longest_consecutive_run_p90 = 3 are shown first — timestamps are UTC)

1) ts-auth-service — mem
   - p95 = 246,497,000 ; p90 = 246,423,200
   - anomaly_count_p90 = 3, anomaly_count_p95 = 2
   - earliest_p90 = 2024-01-25T10:56:00Z
   - max_in_window = 246,955,000 ; min_in_window = 244,502,000
   - longest consecutive run >= p90 = 3

2) ts-order-service — latency-50
   - p95 = 0.024591 ; p90 = 0.013888
   - anomaly_count_p90 = 3, anomaly_count_p95 = 2
   - earliest_p90 = 2024-01-25T10:34:00Z
   - max_in_window = 0.307912 ; min_in_window = 0.008295
   - longest run = 3

3) ts-order-service — mem
   - p95 = 610,514,900 ; p90 = 467,124,200
   - anomaly_count_p90 = 3, anomaly_count_p95 = 2
   - earliest_p90 = 2024-01-25T10:34:00Z
   - max_in_window = 614,754,900 ; min_in_window = 252,017,300
   - longest run = 3

4) ts-order-service — socket
   - p95 = 29.98333 ; p90 = 23.64333
   - anomaly_count_p90 = 3, anomaly_count_p95 = 2
   - earliest_p90 = 2024-01-25T10:34:00Z
   - max_in_window = 30.16667 ; min_in_window = 11.51667
   - longest run = 3

5) ts-route-service — mem
   - p95 = 696,646,800 ; p90 = 641,560,900
   - anomaly_count_p90 = 3, anomaly_count_p95 = 2
   - earliest_p90 = 2024-01-25T10:56:00Z
   - max_in_window = 734,000,200 ; min_in_window = 260,100,100
   - longest run = 3

6) ts-train-service — latency-50
   - p95 = 0.005896 ; p90 = 0.005466
   - anomaly_count_p90 = 3, anomaly_count_p95 = 2
   - earliest_p90 = 2024-01-25T10:47:00Z
   - max_in_window = 0.00662 ; min_in_window = 0.004101
   - longest run = 3

7) ts-train-service — latency-90
   - p95 = 0.011355 ; p90 = 0.009385
   - anomaly_count_p90 = 3, anomaly_count_p95 = 2
   - earliest_p90 = 2024-01-25T10:47:00Z
   - max_in_window = 0.028254 ; min_in_window = 0.008705
   - longest run = 3

Other notable groups (longest run = 2)
- ts-order-service: cpu, latency-90
- ts-route-service: cpu, diskio, latency-50
- ts-train-service: cpu, mem
- ts-travel-service: cpu, latency-90, mem, socket
(These typically show anomaly_count_p90 = 3 and anomaly_count_p95 = 2, and earliest_p90 around 10:34–10:56 UTC.)

Interpretation and recommendation
- Several services exhibited short sustained exceedances of their global p90 threshold (up to 3 consecutive points) in memory, latency, socket and CPU metrics around ~10:34–10:56 UTC.
- The pattern (anomaly_count_p90 = 3 and anomaly_count_p95 = 2) suggests brief spikes rather than long-running degradation.
- Recommended next steps:
  1. Inspect the raw time-aligned metric points for the listed groups (especially order-service mem/socket/cpu and auth/route mem) to confirm impact and correlate with logs/traces.
  2. Check whether the 3-point runs correspond to user-visible errors or service slowdowns.
  3. If confirmed, investigate resource pressure (memory growth, socket usage) and service traces at those timestamps.

If you want, I can (a) show the windowed timestamps/values for any specific group above, or (b) correlate these anomaly times with trace/log events.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id    kpi_name           p95           p90  anomaly_count_p95  anomaly_count_p90 earliest_anomaly_ts_p90  max_value_in_window  min_value_in_window  longest_consecutive_run_p90
4     ts-auth-service         mem  2.464970e+08  2.464232e+08                  2                  3    2024-01-25T10:56:00Z         2.469550e+08         2.445020e+08                            3
9    ts-order-service  latency-50  2.459100e-02  1.388800e-02                  2                  3    2024-01-25T10:34:00Z         3.079120e-01         8.295000e-03                            3
11   ts-order-service         mem  6.105149e+08  4.671242e+08                  2                  3    2024-01-25T10:34:00Z         6.147549e+08         2.520173e+08                            3
12   ts-order-service      socket  2.998333e+01  2.364333e+01                  2                  3    2024-01-25T10:34:00Z         3.016667e+01         1.151667e+01                            3
18   ts-route-service         mem  6.966468e+08  6.415609e+08                  2                  3    2024-01-25T10:56:00Z         7.340002e+08         2.601001e+08                            3
23   ts-train-service  latency-50  5.896000e-03  5.466000e-03                  2                  3    2024-01-25T10:47:00Z         6.620000e-03         4.101000e-03                            3
24   ts-train-service  latency-90  1.135500e-02  9.385000e-03                  2                  3    2024-01-25T10:47:00Z         2.825400e-02         8.705000e-03                            3
7    ts-order-service         cpu  1.210774e+01  5.517637e+00                  2                  3    2024-01-25T10:34:00Z         3.052660e+01         1.669325e+00                            2
10   ts-order-service  latency-90  1.169080e-01  7.147100e-02                  2                  3    2024-01-25T10:34:00Z         1.884046e+00         2.084300e-02                            2
14   ts-route-service         cpu  4.896964e+01  4.890748e+01                  2                  3    2024-01-25T10:52:00Z         4.909843e+01         3.405810e+00                            2
15   ts-route-service      diskio  4.364161e+07  4.291957e+07                  2                  3    2024-01-25T10:52:00Z         4.411491e+07         4.669041e+04                            2
16   ts-route-service  latency-50  2.775200e-02  2.359600e-02                  2                  3    2024-01-25T10:47:00Z         4.578400e-02         3.385000e-03                            2
21   ts-train-service         cpu  5.968902e+00  3.829501e+00                  2                  3    2024-01-25T10:47:00Z         1.893182e+01         2.190027e+00                            2
25   ts-train-service         mem  2.574063e+08  2.569649e+08                  2                  3    2024-01-25T10:48:00Z         2.601675e+08         2.522141e+08                            2
28  ts-travel-service         cpu  3.386332e+00  3.250038e+00                  2                  3    2024-01-25T10:35:00Z         3.545355e+00         1.904188e+00                            2
31  ts-travel-service  latency-90  7.071860e-01  6.905000e-01                  2                  3    2024-01-25T10:47:00Z         7.214580e-01         9.135700e-02                            2
32  ts-travel-service         mem  2.714688e+08  2.714445e+08                  2                  3    2024-01-25T10:55:00Z         2.714786e+08         2.704248e+08                            2
33  ts-travel-service      socket  1.910667e+01  1.872000e+01                  2                  3    2024-01-25T10:49:00Z         2.036667e+01         1.325000e+01                            2
0     ts-auth-service         cpu  1.342455e+01  1.317192e+01                  2                  3    2024-01-25T10:40:00Z         1.398359e+01         9.739225e+00                            1
1     ts-auth-service      diskio  5.996113e+04  5.804265e+04                  2                  3    2024-01-25T10:35:00Z         6.347966e+04         3.862706e+04                            1```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np

# Reuse trace_df and candidates if present; otherwise load from 'trace.csv'
try:
    trace_df
except NameError:
    trace_df = pd.read_csv("trace.csv")

candidates = ["ts-auth-service", "ts-order-service", "ts-route-service", "ts-train-service", "ts-travel-service"]

# Incident window (UTC)
window_start = pd.to_datetime("2024-01-25 10:31:10", utc=True)
window_end   = pd.to_datetime("2024-01-25 11:01:10", utc=True)

# Ensure timestamp is parsed as UTC datetime and value numeric
if "timestamp" in trace_df.columns and not pd.api.types.is_datetime64_any_dtype(trace_df["timestamp"]):
    trace_df["timestamp"] = pd.to_datetime(trace_df["timestamp"], unit="s", utc=True)
else:
    if "timestamp" in trace_df.columns:
        if trace_df["timestamp"].dt.tz is None:
            trace_df["timestamp"] = trace_df["timestamp"].dt.tz_localize("UTC")
        else:
            trace_df["timestamp"] = trace_df["timestamp"].dt.tz_convert("UTC")

trace_df["_v"] = pd.to_numeric(trace_df["value"], errors="coerce")

# Filter to candidate services for threshold computation
trace_cand_full = trace_df[trace_df["cmdb_id"].isin(candidates) & trace_df["_v"].notna()].copy()

# Compute global thresholds (p95, p90) per (cmdb_id, trace_name) using full series
thresholds = (
    trace_cand_full
    .groupby(["cmdb_id", "trace_name"], observed=True)["_v"]
    .agg(p95=lambda x: x.quantile(0.95), p90=lambda x: x.quantile(0.90))
    .reset_index()
)

# Filter trace rows to the incident window
trace_window = trace_df[
    (trace_df["cmdb_id"].isin(candidates)) &
    (trace_df["timestamp"] >= window_start) &
    (trace_df["timestamp"] <= window_end) &
    (trace_df["_v"].notna())
].copy()

if trace_window.shape[0] == 0:
    result_df = pd.DataFrame([{"message": "No trace rows for candidate services in the incident window."}])
else:
    # Merge global thresholds into window rows
    merged = trace_window.merge(thresholds, on=["cmdb_id", "trace_name"], how="left")
    # Drop groups without thresholds (shouldn't happen but safe)
    merged = merged[merged["p95"].notna() & merged["p90"].notna()].copy()
    if merged.shape[0] == 0:
        result_df = pd.DataFrame([{"message": "No groups with precomputed thresholds present in the window."}])
    else:
        # Flag error_rate trace_names (case-insensitive)
        merged["is_error_rate_name"] = merged["trace_name"].str.contains("error_rate", case=False, na=False)
        # Flags for threshold breaches
        merged["is_p95"] = merged["_v"] >= merged["p95"]
        merged["is_p90"] = merged["_v"] >= merged["p90"]
        merged["is_error_point"] = (merged["is_error_rate_name"]) & (merged["_v"] > 0)

        # Aggregate per group
        agg = merged.groupby(["cmdb_id", "trace_name"], observed=True).agg(
            global_p95=pd.NamedAgg(column="p95", aggfunc="first"),
            global_p90=pd.NamedAgg(column="p90", aggfunc="first"),
            anomaly_count_p95=pd.NamedAgg(column="is_p95", aggfunc="sum"),
            anomaly_count_p90=pd.NamedAgg(column="is_p90", aggfunc="sum"),
            error_rate_points=pd.NamedAgg(column="is_error_point", aggfunc="sum"),
            earliest_timestamp_in_window=pd.NamedAgg(column="timestamp", aggfunc="min"),
            mean_in_window=pd.NamedAgg(column="_v", aggfunc="mean"),
            max_in_window=pd.NamedAgg(column="_v", aggfunc="max"),
            min_in_window=pd.NamedAgg(column="_v", aggfunc="min"),
            count_in_window=pd.NamedAgg(column="_v", aggfunc="count"),
        ).reset_index()

        # Keep groups where anomaly_count_p90 > 0 OR error_rate_points > 0
        filtered = agg[(agg["anomaly_count_p90"] > 0) | (agg["error_rate_points"] > 0)].copy()

        if filtered.shape[0] == 0:
            result_df = pd.DataFrame([{"message": "No groups with p90 breaches or error_rate>0 in the window."}])
        else:
            # Format earliest timestamp to ISO UTC string
            filtered["earliest_timestamp_in_window"] = filtered["earliest_timestamp_in_window"].dt.strftime("%Y-%m-%dT%H:%M:%SZ")
            # Round numeric columns for compactness
            for c in ["global_p95", "global_p90", "mean_in_window", "max_in_window", "min_in_window"]:
                filtered[c] = filtered[c].round(6)
            # Select and order required columns
            result_df = filtered[[
                "cmdb_id", "trace_name", "global_p95", "global_p90",
                "anomaly_count_p95", "anomaly_count_p90", "error_rate_points",
                "earliest_timestamp_in_window", "mean_in_window", "max_in_window"
            ]].sort_values(["anomaly_count_p90", "anomaly_count_p95"], ascending=[False, False]).head(20)

# Final compact output
result_df

```
Out[5]:
```
Summary of trace analysis for the incident window (2024-01-25T10:31:10 → 2024-01-25T11:01:10 UTC) for the five candidate services:

- What was found:
  - The returned groups are almost entirely trace.*.error_rate series for the candidate services (auth, order, route, train, travel).
  - For these groups global p90 and p95 (computed on the full series) are 0.0. As a result every window point (including zeros) meets >= p90 and >= p95 and shows anomaly_count_p90 = 25 and anomaly_count_p95 = 25 in the window.
  - However, error_rate_points (count of window points where trace_name contains "error_rate" and value > 0) = 0 for all these groups. mean_in_window and max_in_window are 0.0. Earliest anomaly timestamps are ~2024-01-25T10:34:00Z.

- Interpretation:
  - These "anomalies" are artifacts caused by zero-valued global thresholds (p90==p95==0). They do NOT indicate non-zero error rates or visible trace errors in the incident window.
  - No trace groups reported actual non-zero error_rate events during the window.

- Recommendation:
  - To avoid these false positives, recompute or filter thresholds to ignore zero baselines (require p90>0), or require anomaly points to exceed thresholds by an absolute delta.
  - If you want to look for real issues, examine non-error trace metrics (duration_p95/duration_mean) and other KPI traces (latency, CPU) for spikes in the same window, or correlate with logs around the timestamps ~10:34–10:56 UTC.

If you want, I can (a) re-run the selection enforcing global_p90>0, (b) scan trace duration metrics for p90 breaches, or (c) show raw windowed values for any specific trace group.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                 trace_name  global_p95  global_p90  anomaly_count_p95  anomaly_count_p90  error_rate_points earliest_timestamp_in_window  mean_in_window  max_in_window
2    ts-auth-service                 trace.from_root.error_rate         0.0         0.0                 25                 25                  0         2024-01-25T10:34:00Z             0.0            0.0
6    ts-auth-service      trace.from_ts-auth-service.error_rate         0.0         0.0                 25                 25                  0         2024-01-25T10:34:00Z             0.0            0.0
10   ts-auth-service        trace.to_ts-auth-service.error_rate         0.0         0.0                 25                 25                  0         2024-01-25T10:34:00Z             0.0            0.0
14  ts-order-service                 trace.from_root.error_rate         0.0         0.0                 25                 25                  0         2024-01-25T10:34:00Z             0.0            0.0
22  ts-order-service     trace.from_ts-order-service.error_rate         0.0         0.0                 25                 25                  0         2024-01-25T10:34:00Z             0.0            0.0
26  ts-order-service  trace.from_ts-preserve-service.error_rate         0.0         0.0                 25                 25                  0         2024-01-25T10:34:00Z             0.0            0.0
30  ts-order-service      trace.from_ts-seat-service.error_rate         0.0         0.0                 25                 25                  0         2024-01-25T10:34:00Z             0.0            0.0
38  ts-order-service    trace.from_ts-travel-service.error_rate         0.0         0.0                 25                 25                  0         2024-01-25T10:34:00Z             0.0            0.0
42  ts-order-service       trace.to_ts-order-service.error_rate         0.0         0.0                 25                 25                  0         2024-01-25T10:34:00Z             0.0            0.0
46  ts-order-service     trace.to_ts-station-service.error_rate         0.0         0.0                 25                 25                  0         2024-01-25T10:34:00Z             0.0            0.0
50  ts-route-service     trace.from_ts-basic-service.error_rate         0.0         0.0                 25                 25                  0         2024-01-25T10:34:00Z             0.0            0.0
54  ts-route-service     trace.from_ts-route-service.error_rate         0.0         0.0                 25                 25                  0         2024-01-25T10:34:00Z             0.0            0.0
58  ts-route-service    trace.from_ts-travel-service.error_rate         0.0         0.0                 25                 25                  0         2024-01-25T10:34:00Z             0.0            0.0
62  ts-route-service   trace.from_ts-travel2-service.error_rate         0.0         0.0                 25                 25                  0         2024-01-25T10:34:00Z             0.0            0.0
66  ts-route-service       trace.to_ts-route-service.error_rate         0.0         0.0                 25                 25                  0         2024-01-25T10:34:00Z             0.0            0.0
70  ts-train-service     trace.from_ts-basic-service.error_rate         0.0         0.0                 25                 25                  0         2024-01-25T10:34:00Z             0.0            0.0
74  ts-train-service     trace.from_ts-train-service.error_rate         0.0         0.0                 25                 25                  0         2024-01-25T10:34:00Z             0.0            0.0
78  ts-train-service    trace.from_ts-travel-service.error_rate         0.0         0.0                 25                 25                  0         2024-01-25T10:34:00Z             0.0            0.0
82  ts-train-service   trace.from_ts-travel2-service.error_rate         0.0         0.0                 25                 25                  0         2024-01-25T10:34:00Z             0.0            0.0
86  ts-train-service       trace.to_ts-train-service.error_rate         0.0         0.0                 25                 25                  0         2024-01-25T10:34:00Z             0.0            0.0```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np

# Reuse trace_df from kernel state; if not present, load it
try:
    trace_df
except NameError:
    trace_df = pd.read_csv("trace.csv")

# Ensure timestamp parsed as UTC from Unix seconds and value numeric
trace_df["timestamp"] = pd.to_datetime(trace_df["timestamp"], unit="s", utc=True)
trace_df["_v"] = pd.to_numeric(trace_df["value"], errors="coerce")

# Selection: rows that mention order service
sel_mask = (
    (trace_df["cmdb_id"] == "ts-order-service")
    | trace_df["trace_name"].str.contains("to_ts-order-service", case=False, na=False)
    | trace_df["trace_name"].str.contains("from_ts-order-service", case=False, na=False)
)
trace_sel_full = trace_df[sel_mask].copy()

# Compute global thresholds (p95, p90) per (cmdb_id, trace_name) using full selected series
thresholds = (
    trace_sel_full[trace_sel_full["_v"].notna()]
    .groupby(["cmdb_id", "trace_name"], observed=True)["_v"]
    .agg(global_p95=lambda x: x.quantile(0.95), global_p90=lambda x: x.quantile(0.90))
    .reset_index()
)

# Incident window
window_start = pd.to_datetime("2024-01-25T10:31:10Z", utc=True)
window_end   = pd.to_datetime("2024-01-25T11:01:10Z", utc=True)

# Filter trace rows to the incident window (but only those that match selection)
trace_window = trace_sel_full[
    (trace_sel_full["timestamp"] >= window_start) & (trace_sel_full["timestamp"] <= window_end)
].copy()

# Focus on trace_name types of interest
types_mask = trace_window["trace_name"].str.contains("duration_mean|duration_p95|row_count", case=False, na=False)
trace_window = trace_window[types_mask].copy()

# Merge global thresholds into windowed rows
merged = trace_window.merge(thresholds, on=["cmdb_id", "trace_name"], how="left")

# Drop rows without thresholds or without numeric values
merged = merged[merged["_v"].notna() & merged["global_p95"].notna() & merged["global_p90"].notna()].copy()

if merged.shape[0] == 0:
    result_df = pd.DataFrame([{"message": "No matching trace groups in window after applying filters."}])
else:
    # Flags for threshold breaches
    merged["is_p95"] = merged["_v"] >= merged["global_p95"]
    merged["is_p90"] = merged["_v"] >= merged["global_p90"]
    # Aggregate per group
    agg = merged.groupby(["cmdb_id", "trace_name"], observed=True).agg(
        global_p95=pd.NamedAgg(column="global_p95", aggfunc="first"),
        global_p90=pd.NamedAgg(column="global_p90", aggfunc="first"),
        anomaly_count_p95=pd.NamedAgg(column="is_p95", aggfunc="sum"),
        anomaly_count_p90=pd.NamedAgg(column="is_p90", aggfunc="sum"),
        earliest_anomaly_ts_p90=pd.NamedAgg(column="timestamp", aggfunc=lambda x: x[x.index[(merged.loc[x.index,"is_p90"])].min()] if (merged.loc[x.index,"is_p90"].any()) else pd.NaT),
        mean_in_window=pd.NamedAgg(column="_v", aggfunc="mean"),
        max_in_window=pd.NamedAgg(column="_v", aggfunc="max"),
        count_in_window=pd.NamedAgg(column="_v", aggfunc="count"),
    ).reset_index()

    # The lambda above for earliest_anomaly_ts_p90 is a bit complex; recompute earliest per group explicitly to be safe
    def compute_earliest_p90(df):
        df_p90 = df[df["is_p90"]]
        if df_p90.shape[0] == 0:
            return pd.NaT
        return df_p90["timestamp"].min()

    earliest = (
        merged[merged["is_p90"]]
        .groupby(["cmdb_id", "trace_name"], observed=True)["timestamp"]
        .min()
        .reset_index()
        .rename(columns={"timestamp": "earliest_anomaly_ts_p90"})
    )

    agg = agg.drop(columns=["earliest_anomaly_ts_p90"], errors="ignore").merge(earliest, on=["cmdb_id", "trace_name"], how="left")

    # Keep only groups with anomaly_count_p90 > 0
    filtered = agg[agg["anomaly_count_p90"] > 0].copy()

    if filtered.shape[0] == 0:
        result_df = pd.DataFrame([{"message": "No groups with anomaly_count_p90 > 0 in the window."}])
    else:
        # Format earliest timestamp to ISO UTC strings or None
        filtered["earliest_anomaly_ts_p90"] = filtered["earliest_anomaly_ts_p90"].dt.strftime("%Y-%m-%dT%H:%M:%SZ")
        # Round numeric columns for compactness
        for c in ["global_p95", "global_p90", "mean_in_window", "max_in_window"]:
            filtered[c] = filtered[c].round(6)
        # Select required columns and sort
        result_df = filtered[[
            "cmdb_id", "trace_name", "global_p95", "global_p90",
            "anomaly_count_p95", "anomaly_count_p90",
            "earliest_anomaly_ts_p90", "mean_in_window", "max_in_window"
        ]].sort_values(["anomaly_count_p90", "anomaly_count_p95"], ascending=[False, False]).head(20)

# Display final compact result
result_df

```
Out[6]:
```
```python
summary = (
    "Trace analysis focused on traces that mention the order service (full series thresholds, window 2024-01-25T10:31:10Z–2024-01-25T11:01:10Z):\n\n"
    "What was detected:\n"
    "- Multiple trace groups related to the order service show threshold breaches in the incident window (anomaly_count_p90 = 3 and anomaly_count_p95 = 2 for many groups).\n"
    "- Affected groups include both row_count and latency traces for calls to/from the order service: e.g.:\n"
    "  * trace.to_ts-order-service.row_count (root, ts-inside-payment-service, ts-preserve-service) — global p90 ≈ 10.6, p95 ≈ 11; window max = 10–11, mean ~4–6.\n"
    "  * trace.from_root.duration_mean / duration_p95 (root and ts-order-service) — global p90 ≈ 0.411–0.443 s, p95 ≈ 0.424–0.446 s; window mean ≈0.35–0.38 s, max ≈0.45–0.48 s.\n"
    "  * Several service-to-order and order-to-service duration metrics (inside-payment, preserve, station, etc.) also breached p90/p95 in the window.\n\n"
    "Timing / scope:\n"
    "- Earliest breaches are clustered around 2024-01-25T10:34:00Z for many groups; other notable earliest times include ~10:42:00Z and ~10:56:00Z depending on the group.\n"
    "- Most groups had 3 window points >= global p90 and 2 points >= global p95, indicating short repeated spikes rather than a prolonged sustained outage.\n\n"
    "Interpretation and recommended next steps:\n"
    "- The order service and callers exhibited brief increases in request volume (row_count) and in trace durations around ~10:34Z (and some at ~10:42Z and ~10:56Z).\n"
    "- These patterns suggest transient load or latency spikes affecting the order request paths. Correlate these timestamps with logs, metric spikes (CPU/memory/socket), and any deployment/infra events.\n"
    "- Prioritize investigating: root→order and service→order row_count spikes and root/from_root duration_p95 spikes (they show the largest max durations ~0.45–0.48s).\n\n"
    "If you want, I can: (a) show the windowed timestamps/values for any specific trace group above, or (b) correlate these trace spikes with metric or log events at the same timestamps."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                         trace_name  global_p95  global_p90  anomaly_count_p95  anomaly_count_p90 earliest_anomaly_ts_p90  mean_in_window  max_in_window
2                        root                trace.to_ts-order-service.row_count   11.000000   10.600000                  3                  3    2024-01-25T10:34:00Z        6.440000      11.000000
5   ts-inside-payment-service                trace.to_ts-order-service.row_count   10.000000    9.400000                  3                  3    2024-01-25T10:34:00Z        4.869565      10.000000
8            ts-order-service                          trace.from_root.row_count   11.000000   10.600000                  3                  3    2024-01-25T10:34:00Z        6.440000      11.000000
11           ts-order-service     trace.from_ts-inside-payment-service.row_count   10.000000    9.400000                  3                  3    2024-01-25T10:34:00Z        4.869565      10.000000
17           ts-order-service           trace.from_ts-preserve-service.row_count    6.000000    5.600000                  3                  3    2024-01-25T10:42:00Z        3.400000       7.000000
32           ts-order-service              trace.to_ts-station-service.row_count   11.000000   10.600000                  3                  3    2024-01-25T10:34:00Z        6.440000      11.000000
35        ts-preserve-service                trace.to_ts-order-service.row_count    6.000000    5.600000                  3                  3    2024-01-25T10:42:00Z        3.400000       7.000000
44         ts-station-service              trace.from_ts-order-service.row_count   11.000000   10.600000                  3                  3    2024-01-25T10:34:00Z        6.440000      11.000000
0                        root            trace.to_ts-order-service.duration_mean    0.424245    0.411314                  2                  3    2024-01-25T10:56:00Z        0.352972       0.446626
1                        root             trace.to_ts-order-service.duration_p95    0.446449    0.443187                  2                  3    2024-01-25T10:34:00Z        0.376435       0.482434
3   ts-inside-payment-service            trace.to_ts-order-service.duration_mean    0.029624    0.014420                  2                  3    2024-01-25T10:34:00Z        0.009325       0.039353
4   ts-inside-payment-service             trace.to_ts-order-service.duration_p95    0.053116    0.045444                  2                  3    2024-01-25T10:34:00Z        0.016516       0.078190
6            ts-order-service                      trace.from_root.duration_mean    0.424245    0.411314                  2                  3    2024-01-25T10:56:00Z        0.352972       0.446626
7            ts-order-service                       trace.from_root.duration_p95    0.446449    0.443187                  2                  3    2024-01-25T10:34:00Z        0.376435       0.482434
9            ts-order-service  trace.from_ts-inside-payment-service.duration_...    0.029624    0.014420                  2                  3    2024-01-25T10:34:00Z        0.009325       0.039353
10           ts-order-service  trace.from_ts-inside-payment-service.duration_p95    0.053116    0.045444                  2                  3    2024-01-25T10:34:00Z        0.016516       0.078190
12           ts-order-service          trace.from_ts-order-service.duration_mean    0.012387    0.012111                  2                  3    2024-01-25T10:34:00Z        0.009361       0.019382
13           ts-order-service           trace.from_ts-order-service.duration_p95    0.009339    0.009055                  2                  3    2024-01-25T10:34:00Z        0.011604       0.097592
14           ts-order-service              trace.from_ts-order-service.row_count  454.800000  437.600000                  2                  3    2024-01-25T10:38:00Z      354.000000     465.000000
15           ts-order-service       trace.from_ts-preserve-service.duration_mean    0.107229    0.067608                  2                  3    2024-01-25T10:34:00Z        0.029029       0.130391```
```

In [ ]:
# In[7]:

import pandas as pd
import numpy as np
import itertools

# Reuse metric_df and log_df from kernel state
# Ensure metric_df and log_df exist and have proper types
if 'metric_df' not in globals():
    metric_df = pd.read_csv("metric.csv")
if 'log_df' not in globals():
    log_df = pd.read_csv("log.csv")

# Parse timestamps as UTC datetimes (Unix seconds) per rules
metric_df["timestamp"] = pd.to_datetime(metric_df["timestamp"], unit="s", utc=True)
log_df["timestamp"] = pd.to_datetime(log_df["timestamp"], unit="s", utc=True)

# Ensure numeric values column and normalized column name
metric_df["_v"] = pd.to_numeric(metric_df["value"], errors="coerce")
log_df["_v"] = pd.to_numeric(log_df["value"], errors="coerce")

# Parameters
service = "ts-order-service"
kpis = ["cpu", "mem", "socket", "latency-50", "latency-90"]
window_start = pd.to_datetime("2024-01-25T10:31:10Z", utc=True)
window_end   = pd.to_datetime("2024-01-25T11:01:10Z", utc=True)

# Compute global thresholds (p95, p90, p5) for the full series (no time filter) for the service
full_series = metric_df[metric_df["cmdb_id"] == service].copy()
thresholds = (
    full_series.groupby("kpi_name", observed=True)["_v"]
    .agg(global_p95=lambda x: x.quantile(0.95),
         global_p90=lambda x: x.quantile(0.90),
         global_p5 =lambda x: x.quantile(0.05))
    .reset_index()
)

# Keep only requested KPIs; if missing, they'll be reported with NaNs
thresholds = thresholds[thresholds["kpi_name"].isin(kpis)].set_index("kpi_name")

# Filter metric points to the incident window for the service and requested KPIs
window_points = metric_df[
    (metric_df["cmdb_id"] == service) &
    (metric_df["kpi_name"].isin(kpis)) &
    (metric_df["timestamp"] >= window_start) &
    (metric_df["timestamp"] <= window_end) &
    (metric_df["_v"].notna())
].copy().sort_values("timestamp")

# Function to compute longest consecutive run >= threshold
def longest_consecutive_run(values, threshold):
    if len(values) == 0:
        return 0
    mask = (values >= threshold).astype(int)
    runs = [sum(1 for _ in group) for val, group in itertools.groupby(mask) if val == 1]
    return int(max(runs)) if runs else 0

# Build result rows for each KPI
rows = []
for k in kpis:
    if k in thresholds.index:
        p95 = float(thresholds.loc[k, "global_p95"])
        p90 = float(thresholds.loc[k, "global_p90"])
        p5  = float(thresholds.loc[k, "global_p5"])
    else:
        p95 = p90 = p5 = np.nan
    grp = window_points[window_points["kpi_name"] == k].copy().sort_values("timestamp")
    vals = grp["_v"].to_numpy()
    times = grp["timestamp"].to_numpy()
    anomaly_count_p95 = int(np.sum(vals >= p95)) if len(vals) > 0 and not np.isnan(p95) else 0
    anomaly_count_p90 = int(np.sum(vals >= p90)) if len(vals) > 0 and not np.isnan(p90) else 0
    earliest_anomaly_ts_p90 = None
    if anomaly_count_p90 > 0:
        idxs = np.where(vals >= p90)[0]
        earliest_anomaly_ts_p90 = pd.to_datetime(times[idxs[0]]).tz_convert("UTC").strftime("%Y-%m-%dT%H:%M:%SZ")
    max_value_in_window = float(np.nan) if len(vals) == 0 else float(np.nanmax(vals))
    min_value_in_window = float(np.nan) if len(vals) == 0 else float(np.nanmin(vals))
    longest_run = longest_consecutive_run(vals, p90) if not np.isnan(p90) else 0
    if np.isnan(p95) or p95 == 0:
        max_severity_ratio = None
    else:
        max_severity_ratio = None if len(vals) == 0 else float(np.max(vals / p95))
    rows.append({
        "source": "metric",
        "cmdb_id": service,
        "kpi_name": k,
        "global_p95": round(p95, 6) if not np.isnan(p95) else np.nan,
        "global_p90": round(p90, 6) if not np.isnan(p90) else np.nan,
        "global_p5":  round(p5, 6)  if not np.isnan(p5)  else np.nan,
        "anomaly_count_p95": anomaly_count_p95,
        "anomaly_count_p90": anomaly_count_p90,
        "earliest_anomaly_ts_p90": earliest_anomaly_ts_p90,
        "max_value_in_window": (round(max_value_in_window,6) if not np.isnan(max_value_in_window) else np.nan),
        "min_value_in_window": (round(min_value_in_window,6) if not np.isnan(min_value_in_window) else np.nan),
        "longest_consecutive_run_p90": longest_run,
        "max_severity_ratio": (round(max_severity_ratio,6) if max_severity_ratio is not None else None),
    })

# Also compute ts-order-service log.row_count stats in the window
log_sel = log_df[
    (log_df["cmdb_id"] == service) &
    (log_df["log_name"] == "log.row_count") &
    (log_df["timestamp"] >= window_start) &
    (log_df["timestamp"] <= window_end) &
    (log_df["_v"].notna())
].sort_values("timestamp")

log_count = int(log_sel.shape[0])
log_mean = float(log_sel["_v"].mean()) if log_count > 0 else np.nan
log_max  = float(log_sel["_v"].max()) if log_count > 0 else np.nan
log_earliest = None
if log_count > 0:
    log_earliest = log_sel["timestamp"].min().strftime("%Y-%m-%dT%H:%M:%SZ")

rows.append({
    "source": "log",
    "cmdb_id": service,
    "kpi_name": "log.row_count",
    "global_p95": np.nan,
    "global_p90": np.nan,
    "global_p5": np.nan,
    "anomaly_count_p95": np.nan,
    "anomaly_count_p90": np.nan,
    "earliest_anomaly_ts_p90": None,
    "max_value_in_window": round(log_max,6) if not np.isnan(log_max) else np.nan,
    "min_value_in_window": round(log_mean,6) if not np.isnan(log_mean) else np.nan,  # repurpose min column to show mean for concise table
    "longest_consecutive_run_p90": log_count,
    "max_severity_ratio": log_earliest,  # repurpose this column to show earliest timestamp for compactness
})

# Create result DataFrame and order rows as requested (KPIs then log row)
result_df = pd.DataFrame(rows)
# Reorder columns for clear presentation
result_df = result_df[[
    "source", "cmdb_id", "kpi_name",
    "global_p95", "global_p90", "global_p5",
    "anomaly_count_p95", "anomaly_count_p90", "earliest_anomaly_ts_p90",
    "max_value_in_window", "min_value_in_window",
    "longest_consecutive_run_p90", "max_severity_ratio"
]]

# Keep only the requested KPI rows plus the log.row_count row (already exactly these)
result_df

```
Out[7]:
```
Summary for ts-order-service (incident window 2024-01-25T10:31:10 → 2024-01-25T11:01:10 UTC). Global thresholds were computed from the full series (p95/p90/p5) and then applied to the window:

Per-KPI findings
- cpu
  - global p95=12.10774, p90=5.517637, p5=1.792011
  - anomaly_count_p95 = 2, anomaly_count_p90 = 3
  - earliest p90 breach = 2024-01-25T10:34:00Z
  - max_in_window = 30.52660, min_in_window = 1.669325
  - longest consecutive run >= p90 = 2
  - max_severity_ratio (max value / p95) = 2.521

- mem
  - global p95 = 610,514,900; p90 = 467,124,200; p5 = 253,811,300
  - anomaly_count_p95 = 2, anomaly_count_p90 = 3
  - earliest p90 breach = 2024-01-25T10:34:00Z
  - max_in_window = 614,754,900; min_in_window = 252,017,300
  - longest consecutive run >= p90 = 3
  - max_severity_ratio ≈ 1.007 (mildly above p95)

- socket
  - global p95 = 29.98333, p90 = 23.64333, p5 = 12.32
  - anomaly_count_p95 = 2, anomaly_count_p90 = 3
  - earliest p90 breach = 2024-01-25T10:34:00Z
  - max_in_window = 30.16667; min_in_window = 11.51667
  - longest consecutive run >= p90 = 3
  - max_severity_ratio ≈ 1.006 (mild)

- latency-50
  - global p95 = 0.024591, p90 = 0.013888, p5 = 0.008649
  - anomaly_count_p95 = 2, anomaly_count_p90 = 3
  - earliest p90 breach = 2024-01-25T10:34:00Z
  - max_in_window = 0.307912; min_in_window = 0.008295
  - longest consecutive run >= p90 = 3
  - max_severity_ratio ≈ 12.52 (large spike relative to p95)

- latency-90
  - global p95 = 0.116908, p90 = 0.071471, p5 = 0.021751
  - anomaly_count_p95 = 2, anomaly_count_p90 = 3
  - earliest p90 breach = 2024-01-25T10:34:00Z
  - max_in_window = 1.884046; min_in_window = 0.020843
  - longest consecutive run >= p90 = 2
  - max_severity_ratio ≈ 16.12 (very large spike relative to p95)

ts-order-service log.row_count in the window
- count_of_points_in_window = 25
- mean_row_count_in_window = 295.96
- max_row_count_in_window = 374.0
- earliest_timestamp_in_window = 2024-01-25T10:34:00Z

Interpretation and recommended actions
- Most KPIs show brief threshold breaches concentrated at ~10:34Z (earliest p90 breaches) and some around 10:47–10:56Z; mem/socket had modest excursions around p95, while CPU had a moderate spike (max ≈ 30.5, ratio ~2.5× p95).
- The most severe anomalies are the latency spikes (latency-50 and latency-90) with very high severity ratios (≈12–16× p95) and multiple consecutive points above p90—these are likely user-impacting.
- The log.row_count shows elevated request volume (mean ≈296, peak 374) at ~10:34Z, which correlates with the KPI/latency spikes—suggesting transient load increase that coincides with latency and CPU/memory/socket activity.
- Next steps: examine traces/logs around 10:34Z for request patterns, inspect service-side latency root causes (thread pools, downstream delays, GC or I/O), and check whether traffic surge caused the latency spikes.

The original code execution output of IPython Kernel is also provided below for reference:

source           cmdb_id       kpi_name    global_p95    global_p90     global_p5  anomaly_count_p95  anomaly_count_p90 earliest_anomaly_ts_p90  max_value_in_window  min_value_in_window  longest_consecutive_run_p90    max_severity_ratio
0  metric  ts-order-service            cpu  1.210774e+01  5.517637e+00  1.792011e+00                2.0                3.0    2024-01-25T10:34:00Z         3.052660e+01         1.669325e+00                            2              2.521246
1  metric  ts-order-service            mem  6.105149e+08  4.671242e+08  2.538113e+08                2.0                3.0    2024-01-25T10:34:00Z         6.147549e+08         2.520173e+08                            3              1.006945
2  metric  ts-order-service         socket  2.998333e+01  2.364333e+01  1.232000e+01                2.0                3.0    2024-01-25T10:34:00Z         3.016667e+01         1.151667e+01                            3              1.006115
3  metric  ts-order-service     latency-50  2.459100e-02  1.388800e-02  8.649000e-03                2.0                3.0    2024-01-25T10:34:00Z         3.079120e-01         8.295000e-03                            3             12.521471
4  metric  ts-order-service     latency-90  1.169080e-01  7.147100e-02  2.175100e-02                2.0                3.0    2024-01-25T10:34:00Z         1.884046e+00         2.084300e-02                            2             16.115659
5     log  ts-order-service  log.row_count           NaN           NaN           NaN                NaN                NaN                    None         3.740000e+02         2.959600e+02                           25  2024-01-25T10:34:00Z```
```